# 3D Physics

In the earlier 3D notebooks, nothing moved unless your code moved it. A ball placed in mid-air stayed in mid-air, and shapes passed straight through each other unless you wrote a handler, as you did in [[80FE9F9A-F226-4BF4-AC1B-D5A4B4F7647E|Interactive 3D]].

A **physics engine** changes that. Switch it on and the scene starts following the rules of the real world: things fall, land, roll, pile up, and knock each other over. You set up the starting positions, and the engine works out everything that happens next.

It takes one line:

```python
scene.set_physics(True)
```

The three shapes below are placed in mid-air. Run the cell and watch what happens. Run it again to replay.

In [ ]:
from codetto import graphics3d

scene = graphics3d.Scene()
scene.set_sky('#87ceeb')
ground = scene.set_ground(length=14, width=14)
ground.set_material(graphics3d.Material.Grass.Bright)
ground.set_tiling(6)

crate = graphics3d.Shapes.Box(width=1.5, height=1.5, depth=1.5)
crate.set_position(-2.5, 5, 0)
crate.set_rotation(x=20, z=30)
crate.set_material(graphics3d.Material.Wood.Oak)
scene.add(crate)

ball = graphics3d.Shapes.Sphere(diameter=1.5, segments=24)
ball.set_position(0, 8, 0)
ball.set_material(graphics3d.Material.Sports.Soccerball)
scene.add(ball)

barrel = graphics3d.Shapes.Cylinder(diameter=1.2, height=2)
barrel.set_position(2.5, 6, 0)
barrel.set_rotation(z=50)
barrel.set_color('#e94560')
scene.add(barrel)

scene.set_physics(True)

# What the Engine Does

With physics on, every shape in the scene becomes a solid object.

- **Gravity** pulls every shape downward, a little faster each moment.
- **Shapes are solid.** They land on the ground and on each other, and they push each other out of the way.
- **The ground never moves.** It is the one thing gravity does not affect.
- **Every shape weighs the same**, whatever its size.

A few things work differently once physics is running:

- Let the engine do the moving. You choose where each shape **starts** with `set_position` and `set_rotation`. Moving a shape with `set_position` afterwards fights against the engine.
- `get_position` still reports where *you* last put a shape, not where the engine has carried it.
- Shapes inside a `Group` are ignored by the engine. They stay where they are, and other shapes pass through them.
- Shapes collide as simple solids. A cone collides as if it were a cylinder, and a torus as if its hole were filled in.

Scenes like the one above do not need `scene.run()`. You only need it when the scene also has handlers for clicks, keys, or frames.

# A Chain Reaction

There is no command for pushing a shape. What you can do is start a shape off balance, and let gravity do the pushing.

The cell below stands up a row of dominoes. All of them are perfectly upright except the first, which starts tilted. It tips over, hits the second, and the rest follows.

Try changing the sliders and running the cell again. What happens when the dominoes are too far apart? What is the smallest tilt that still starts the chain?

In [ ]:
DOMINOES = 12 #@param {type:"slider", min:4, max:20, step:1}
SPACING = 1 #@param {type:"slider", min:0.6, max:2.4, step:0.2}
TILT = 20 #@param {type:"slider", min:0, max:40, step:5}

scene = graphics3d.Scene()
scene.set_sky('#1a1a2e')
ground = scene.set_ground(length=10, width=60)
ground.set_material(graphics3d.Material.WoodFloor.PinePlanks)
ground.set_tiling(10)

colors = ['#e94560', '#f5a623', '#f8e71c', '#44cc88', '#4a90d9', '#c77dff']
start = -(DOMINOES - 1) * SPACING / 2

for i in range(DOMINOES):
  domino = graphics3d.Shapes.Box(width=0.25, height=2, depth=1)
  domino.set_position(start + i * SPACING, 1, 0)
  domino.set_color(colors[i % len(colors)])
  if i == 0:
    domino.set_position(start, 1.1, 0)
    domino.set_rotation(z=-TILT)
  scene.add(domino)

scene.camera.set_position(0, 6, -16)
scene.set_physics(True)

# Changing Gravity

Gravity is written as three numbers, like a position: how hard it pulls along `x`, `y`, and `z`. Normal Earth gravity is `(0, -9.81, 0)`. That is no pull sideways, and a pull of 9.81 **down** the y axis.

You can choose the gravity when you switch physics on:

```python
scene.set_physics(True, gravity=(0, -1.62, 0))    # the Moon
```

| Place | Gravity |
|---|---|
| Earth | 9.81 |
| The Moon | 1.62 |
| Mars | 3.72 |
| Jupiter | 24.79 |
| Deep space | 0 |

Pick a place from the dropdown and run the cell. The balls always start at the same heights, so watch how long they take to land.

In [ ]:
PLACE = "Moon" #@param ["Earth", "Moon", "Mars", "Jupiter", "Deep space"]

gravity = {
  'Earth': 9.81,
  'Moon': 1.62,
  'Mars': 3.72,
  'Jupiter': 24.79,
  'Deep space': 0,
}

scene = graphics3d.Scene()
scene.set_sky(graphics3d.Sky.DEEP_SPACE)
ground = scene.set_ground(length=14, width=14)
ground.set_material(graphics3d.Material.Gravel.LightGray)
ground.set_tiling(4)

planets = [
  graphics3d.Material.Planets.Mercury,
  graphics3d.Material.Planets.Venus,
  graphics3d.Material.Planets.Earth,
  graphics3d.Material.Planets.Mars,
  graphics3d.Material.Planets.Jupiter,
]

for i in range(5):
  ball = graphics3d.Shapes.Sphere(diameter=1.2, segments=24)
  ball.set_position(i * 2 - 4, 3 + i * 1.5, 0)
  ball.set_material(planets[i])
  scene.add(ball)

scene.camera.set_position(0, 6, -16).look_at(0, 4, 0)
scene.set_physics(True, gravity=(0, -gravity[PLACE], 0))

print(f'{PLACE}: gravity is {gravity[PLACE]}')

# Pull Out a Block

Physics and events work together. The tower below is balanced and would stand forever. Every piece has a click handler that removes it:

```python
piece.on_click(piece.remove)
```

`piece.remove` is already a function that needs no extra information, so it can be handed straight to `on_click`.

Because this scene uses click handlers, it ends with `scene.run()`. Run the cell, click a piece, and see what the rest of the tower does. Which pieces can you remove without the ball reaching the ground? Press **Stop** (■) when you are done.

In [ ]:
scene = graphics3d.Scene()
scene.set_sky('#87ceeb')
ground = scene.set_ground(length=14, width=14)
ground.set_material(graphics3d.Material.Grass.Bright)
ground.set_tiling(6)

pieces = []

for level in range(3):
  base = level * 2.4

  for x in [-1.2, 1.2]:
    pillar = graphics3d.Shapes.Box(width=0.5, height=2, depth=1.5)
    pillar.set_position(x, base + 1, 0)
    pillar.set_material(graphics3d.Material.Bricks.DarkClay)
    pieces.append(pillar)

  plank = graphics3d.Shapes.Box(width=3.4, height=0.4, depth=1.5)
  plank.set_position(0, base + 2.2, 0)
  plank.set_material(graphics3d.Material.Wood.Oak)
  pieces.append(plank)

for piece in pieces:
  piece.on_click(piece.remove)
  scene.add(piece)

ball = graphics3d.Shapes.Sphere(diameter=1, segments=24)
ball.set_position(0, 7.7, 0)
ball.set_material(graphics3d.Material.Sports.Tennis)
scene.add(ball)

scene.camera.set_position(6, 6, -14).look_at(0, 3.5, 0)
scene.set_physics(True)
scene.run()

# Putting It Together: Knock It Down

Gravity does not have to stay the same, and it does not have to point down. While physics is on, `scene.set_gravity(x, y, z)` changes it instantly. A pull along `x` acts like a strong wind blowing everything sideways.

You can also add new shapes while the scene is running. They join in as soon as they are added.

This last scene stacks a pyramid of cans and gives you the controls:

| Key | What it does |
|---|---|
| **Space** | Drops a ball from above |
| **Left** / **Right** | Tilts gravity to that side |
| **Down** | Puts gravity back to normal |
| **Up** | Turns gravity upside down |

Ideas to try:

- Make the pyramid taller by changing `ROWS`.
- Drop boxes or barrels in place of balls.
- Add a key that switches to Moon gravity.
- Play a sound from `samples.Sounds` each time a ball is dropped.
- Build something of your own to knock down: a wall, a tower, a line of dominoes.

In [ ]:
import random
from functools import partial

ROWS = 4

scene = graphics3d.Scene()
scene.set_sky(graphics3d.Sky.PURE_SKY)
ground = scene.set_ground(length=30, width=30)
ground.set_material(graphics3d.Material.Grass.Bright)
ground.set_tiling(12)

# A pyramid of cans: ROWS on the bottom, one fewer on each row above
for row in range(ROWS):
  cans = ROWS - row
  for i in range(cans):
    can = graphics3d.Shapes.Cylinder(diameter=0.9, height=1)
    can.set_position((i - (cans - 1) / 2) * 1.0, 0.5 + row * 1.0, 0)
    can.set_color('#e94560' if row % 2 == 0 else '#ffffff')
    scene.add(can)

hud = scene.get_context('2d')

def show(message):
  hud.clear()
  hud.fill_style = 'rgba(0, 0, 0, 0.5)'
  hud.fill_rect(10, 10, 300, 44)
  hud.fill_style = '#ffffff'
  hud.font = '20px sans-serif'
  hud.fill_text(message, 22, 39)

def change_gravity(x, y, message):
  scene.set_gravity(x, y, 0)
  show(message)

def drop_ball():
  ball = graphics3d.Shapes.Sphere(diameter=0.9, segments=16)
  ball.set_position(random.uniform(-2, 2), 9, 0)
  ball.set_color(random.choice(['#f5a623', '#4a90d9', '#44cc88', '#c77dff']))
  scene.add(ball)

scene.on_key(graphics3d.Key.SPACE, drop_ball)
scene.on_key(graphics3d.Key.LEFT, partial(change_gravity, -3, -9.81, 'Gravity: tilted left'))
scene.on_key(graphics3d.Key.RIGHT, partial(change_gravity, 3, -9.81, 'Gravity: tilted right'))
scene.on_key(graphics3d.Key.DOWN, partial(change_gravity, 0, -9.81, 'Gravity: normal'))
scene.on_key(graphics3d.Key.UP, partial(change_gravity, 0, 2, 'Gravity: upside down'))

scene.camera.set_position(0, 4, -13).look_at(0, 2, 0)
scene.set_physics(True)
show('Gravity: normal')
scene.run()

# Check Your Understanding